# 03 WeightWatcher Analysis

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


WeightWatcher spectral analysis using canonical schema-v2 scientific runs only.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
specs=[]; projs=[]
for r in runs:
    art=load_run_artifacts(Path(r['run_dir'])); s=art['spectral'].assign(seed=r['seed'],pair_id=r['pair_id'],optimizer_family=r['optimizer_family'],optimizer_label=r['optimizer_label'],optimizer_raw=r['optimizer_raw'],valid_for_science=True,scientific_schema_version=r['manifest'].get('scientific_schema_version'))
    specs.append(s); p=art['projection']
    if not p.empty: projs.append(p.assign(seed=r['seed']))
spectral=pd.concat(specs,ignore_index=True); spectral.to_csv(ANALYSIS_DIR/'spectral_records_scientific.csv',index=False)
valid=spectral[(spectral.spectral_estimator.astype(str).str.lower()=='weightwatcher') & (spectral.valid_for_science) & (spectral.scientific_schema_version.astype(float)>=2)]
validity=spectral.groupby(['optimizer_raw','seed']).agg(rows=('alpha','size'),weightwatcher_rows=('valid_weightwatcher','sum')).reset_index(); validity.to_csv(ANALYSIS_DIR/'spectral_validity_audit.csv',index=False); display(validity)
proj_layers=valid[valid.layer_group=='projected_transformer_matrix'].copy(); proj_layers['alpha_distance_to_2']=(proj_layers['alpha']-2).abs()
summary=proj_layers.groupby(['optimizer_family','seed','tokens_seen']).agg(median_alpha=('alpha','median'),q25_alpha=('alpha',lambda s:s.quantile(.25)),q75_alpha=('alpha',lambda s:s.quantile(.75)),eligible_matrices=('alpha','count')).reset_index(); summary.to_csv(ANALYSIS_DIR/'run_snapshot_alpha_summary.csv',index=False)
proj_layers.groupby(['optimizer_family','layer_name','tokens_seen']).agg(median_alpha=('alpha','median'),q25=('alpha',lambda s:s.quantile(.25)),q75=('alpha',lambda s:s.quantile(.75))).reset_index().to_csv(ANALYSIS_DIR/'projected_layer_alpha_summary.csv',index=False)
for y,f in [('alpha','alpha_trajectories_by_layer.png'),('median_alpha','median_projected_alpha_vs_tokens.png'),('alpha_distance_to_2','distance_to_alpha_2_vs_tokens.png')]:
    fig,ax=plt.subplots(figsize=(7,4))
    data=proj_layers if y!='median_alpha' else summary
    for fam,g in data.groupby('optimizer_family'):
        for _,gg in g.groupby(['seed'] if y=='median_alpha' else ['seed','layer_name']): ax.plot(gg.tokens_seen, gg[y], alpha=.2, color=OPTIMIZER_COLORS[fam])
        mean=g.groupby('tokens_seen')[y].mean(); ax.plot(mean.index,mean.values,lw=3,label=OPTIMIZER_LABELS[fam],color=OPTIMIZER_COLORS[fam])
    ax.legend(); ax.set_xlabel('tokens_seen'); ax.set_ylabel(y); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/f); plt.close(fig)
fig,ax=plt.subplots();
for fam,g in summary.groupby('optimizer_family'): ax.plot(g.tokens_seen,g.median_alpha,'.',label=OPTIMIZER_LABELS[fam])
for p in projs:
    for x in p.tokens_seen.dropna().unique(): ax.axvline(x,color='gray',alpha=.1)
ax.legend(); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'projection_markers_on_alpha.png'); plt.close(fig)
fig,ax=plt.subplots(); ax.scatter(proj_layers.get('num_evals'), proj_layers.get('D')); ax.set_xlabel('num_evals'); ax.set_ylabel('D / KS statistic'); fig.tight_layout(); fig.savefig(ANALYSIS_DIR/'spectral_fit_quality.png'); plt.close(fig)
print('Finite-size warning: level-0 width-64 style models can produce noisy layer-level power-law estimates; aggregate within run/snapshot before comparing seeds.')
